# init-process-group-nccl — ex2: @contextmanager dist_session: destroy guaranteed on exception

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `init-process-group-nccl`. Running the final beacon cell reports progress against the `Distributed: init_process_group nccl` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: init_process_group nccl` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`init-process-group-nccl`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "init-process-group-nccl"
DD_SUBTOPIC = "Distributed: init_process_group nccl"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `init_process_group` wrapped in a context manager

Ex1 called `init_process_group` + `destroy_process_group` by hand. The footgun: if anything between init and destroy raises (loss explodes, OOM, data loader crashes), `destroy` is skipped — the rendezvous port stays bound and the NEXT training run hangs on init.

Context-manager wrap fixes it with `try/finally`:

```python
from contextlib import contextmanager

@contextmanager
def dist_session(rank, world_size, port, backend='gloo'):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend=backend, rank=rank,
                            world_size=world_size)
    try:
        yield
    finally:
        dist.destroy_process_group()
```

**Why `finally`, not `try/except`.** A `try/except` swallows the exception. We want the exception to propagate AND destroy to run. `finally` is the only construct that guarantees both.

**Where this pattern lives in real code.** `torch.distributed.run` (the `torchrun` launcher) wraps every worker in this exact pattern internally. Hand-rolling it for notebook-launched workers gives you the same crash-resilience.

### Exercise 2 — @contextmanager dist_session: destroy guaranteed on exception

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `@contextlib.contextmanager` + `try/finally` to wrap `init_process_group` and `destroy_process_group` so destroy is guaranteed even when the body raises.
> Keywords: contextmanager, init_process_group, destroy_process_group, try-finally
> ```

**KCs targeted:** `contextmanager-init-destroy`, `destroy-runs-even-on-exception`

Implement `ex2_dist_session(rank, world_size, port, dist_module, backend='gloo')`, a context manager. Required behavior:

1. Decorate with `@contextlib.contextmanager`.
2. BEFORE the `yield`: set `os.environ['MASTER_ADDR'] = '127.0.0.1'` and `os.environ['MASTER_PORT'] = str(port)`. Call `dist_module.init_process_group(backend=backend, rank=rank, world_size=world_size)`.
3. Wrap the `yield` in `try / finally`. The `finally` block MUST call `dist_module.destroy_process_group()`, with NO conditional guards — destroy always runs, even when the body raised.
4. `yield` no value (a bare `yield`) — the context manager exists for its side effect.

Input: `rank`, `world_size`, `port` — ints; `dist_module` — torch.distributed or mock; `backend` — str, default `'gloo'`.
Yields: nothing.

The test runs the context manager (a) normally, (b) with an exception inside the `with` body. In both cases destroy must have been called exactly once.

In [ ]:
import contextlib
import os

@contextlib.contextmanager
def ex2_dist_session(rank: int, world_size: int, port: int, dist_module, backend: str = 'gloo'):
    """Init + destroy process group with try/finally; yields nothing."""
    raise NotImplementedError()
    yield  # unreachable — keeps generator framing


def _test_ex2():

    import threading
    import types as _types
    import torch as _t_for_fake

    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'

    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            self.scratch = {}
            self.tls = threading.local()
            self.results = [None] * world_size
            # send/recv mailbox keyed by (src, dst)
            self.mailbox = {}
            self.mailbox_cv = threading.Condition(self.lock)
        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['ar']
            if op == 'SUM':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced + x
            elif op == 'MAX':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.maximum(reduced, x)
            elif op == 'MIN':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.minimum(reduced, x)
            elif op == 'PROD':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced * x
            else:
                raise ValueError(f'unknown fake op {op!r}')
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()
        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['rd']
                if op == 'SUM':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced + x
                elif op == 'MAX':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.maximum(reduced, x)
                elif op == 'MIN':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.minimum(reduced, x)
                elif op == 'PROD':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced * x
                else:
                    raise ValueError(f'unknown fake op {op!r}')
                tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()
        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()
        def barrier_op(self):
            self.barrier.wait()
        def send(self, tensor, dst):
            rank = self.tls.rank
            with self.mailbox_cv:
                self.mailbox.setdefault((rank, dst), []).append(tensor.detach().clone())
                self.mailbox_cv.notify_all()
        def recv(self, tensor, src):
            rank = self.tls.rank
            with self.mailbox_cv:
                while not self.mailbox.get((src, rank)):
                    self.mailbox_cv.wait(timeout=10)
                payload = self.mailbox[(src, rank)].pop(0)
            tensor.copy_(payload)

    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size
        def _runner(rank):
            world.tls.rank = rank
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.send = lambda tensor, dst: world.send(tensor, dst)
            fake_dist.recv = lambda tensor, src: world.recv(tensor, src)
            fake_dist.init_process_group = lambda **kw: world.scratch.setdefault('_init_calls', []).append(kw)
            fake_dist.destroy_process_group = lambda: world.scratch.setdefault('_destroy_calls', []).append(rank)
            try:
                worker_fn(rank, world_size, fake_dist, world)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())
        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world


    import os

    # Normal-path: body runs, destroy fires.
    def _worker(rank, world_size, dist_module, world):
        with ex2_dist_session(rank, world_size, 29610, dist_module):
            # Inside body — group is up; mock barrier should succeed.
            dist_module.barrier()
            world.results[rank] = 'normal-path-ok'

    w = _run_fake_world(_worker, 3)
    for r in range(3):
        assert w.results[r] == 'normal-path-ok', f'rank {r}: body did not run'

    # init_process_group called once per rank (3 inits total).
    init_calls = w.scratch.get('_init_calls', [])
    assert len(init_calls) == 3, f'expected 3 init calls, got {len(init_calls)}'
    # destroy_process_group called once per rank (3 destroys total).
    destroy_calls = w.scratch.get('_destroy_calls', [])
    assert len(destroy_calls) == 3, f'expected 3 destroy calls, got {len(destroy_calls)}'
    assert sorted(destroy_calls) == [0, 1, 2], f'destroy missed ranks: {destroy_calls}'

    # Env vars set.
    assert os.environ.get('MASTER_ADDR') == '127.0.0.1', 'MASTER_ADDR not set'
    assert os.environ.get('MASTER_PORT') == '29610', f"MASTER_PORT got {os.environ.get('MASTER_PORT')!r}"

    # Exception path: destroy STILL runs.
    class _BoomError(RuntimeError):
        pass

    def _worker_boom(rank, world_size, dist_module, world):
        raised = False
        try:
            with ex2_dist_session(rank, world_size, 29611, dist_module):
                raise _BoomError(f'rank {rank} simulated crash')
        except _BoomError:
            raised = True
        world.results[rank] = raised

    w_boom = _run_fake_world(_worker_boom, 2)
    for r in range(2):
        assert w_boom.results[r] is True, f'rank {r}: exception did NOT propagate out of dist_session'
    # destroy still called for every rank despite the exception.
    destroy_calls_boom = w_boom.scratch.get('_destroy_calls', [])
    assert sorted(destroy_calls_boom) == [0, 1], (
        f'exception path: destroy_process_group must run for every rank, got {destroy_calls_boom}.  '
        f'Did you forget try/finally?'
    )
    init_calls_boom = w_boom.scratch.get('_init_calls', [])
    assert len(init_calls_boom) == 2, f'expected 2 init calls, got {len(init_calls_boom)}'

    # Backend kwarg is threaded through.
    def _worker_backend(rank, world_size, dist_module, world):
        with ex2_dist_session(rank, world_size, 29612, dist_module, backend='nccl'):
            pass
        world.results[rank] = 'ok'

    w_b = _run_fake_world(_worker_backend, 2)
    init_calls_b = w_b.scratch.get('_init_calls', [])
    assert all(c.get('backend') == 'nccl' for c in init_calls_b), (
        f'backend kwarg not threaded: {init_calls_b}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
import contextlib
import os

@contextlib.contextmanager
def ex2_dist_session(rank: int, world_size: int, port: int, dist_module, backend: str = 'gloo'):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist_module.init_process_group(backend=backend, rank=rank, world_size=world_size)
    try:
        yield
    finally:
        dist_module.destroy_process_group()
```

**Why `finally`, not `try/except`.** `try/except` would catch the exception and swallow it — wrong. `finally` runs the cleanup AND re-raises whatever was in flight. The two-line discipline of `init` → `try: yield ... finally: destroy` is the entire pattern.

**Env vars before init.** `init_process_group` reads `MASTER_ADDR` and `MASTER_PORT` from `os.environ` if not passed as kwargs. Setting them BEFORE the init call (not after) is mandatory.

**Real-world equivalent.** `torchrun` (the elastic launcher) wraps every worker in this exact pattern internally. For notebook-launched workers, hand-rolling the context manager gives you the same crash-resilience without launching from the CLI.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()